# 🔗 05 - Relationships and Correlation

Notebook 04 compared a numeric column across **groups**. This notebook compares two **numeric** columns with each other: when one goes up, does the other tend to go up, go down, or do nothing?

The tool is **correlation**: one number from −1 to 1. It's useful, it's everywhere, and it's easy to over-trust. This notebook ends with four datasets that share the same correlation and look nothing alike, which is the reason Module 08 exists.

## ✅ Learning Goals
- Describe a relationship by its **direction** and **strength**
- Compute Pearson correlation with `.corr()`, for one pair or a whole table
- Find the strongest pairs in a correlation matrix
- Choose **Spearman** correlation for ranked, curved, or outlier-heavy data
- Explain why r ≈ 0 doesn't mean "no relationship," and why correlation isn't causation

In [1]:
import numpy as np
import pandas as pd

poke = pd.read_csv('pokemon.csv')
titanic = pd.read_csv('titanic.csv')

stats = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

## 1. Direction and Strength
| Relationship | Meaning | Example |
|---|---|---|
| **Positive** | as one rises, the other tends to rise | study hours and exam score |
| **Negative** | as one rises, the other tends to fall | car age and resale price |
| **None** | knowing one tells you nothing about the other | shoe size and exam score |

"Tends to" is doing a lot of work. A positive relationship means **on average**, not for every single row.

### Covariance: The Right Idea, Awkward Units
**Covariance** measures whether two columns move together. It's positive when they rise together and negative when one rises as the other falls:

In [2]:
print('Covariance of Attack and Defense:', round(poke['Attack'].cov(poke['Defense']), 1))
print('Covariance of Attack/10 and Defense:', round((poke['Attack'] / 10).cov(poke['Defense']), 1))

Covariance of Attack and Defense: 444.0
Covariance of Attack/10 and Defense: 44.4


Dividing Attack by 10 changed the covariance by 10×, even though the relationship didn't change at all. Covariance depends on the **units**, so its size is hard to interpret. Correlation fixes that.

## 2. Pearson Correlation
**Pearson's r** is covariance rescaled to always land between −1 and 1, with no units:

In [3]:
print('r(Attack, Defense):    ', round(poke['Attack'].corr(poke['Defense']), 3))
print('r(Attack/10, Defense): ', round((poke['Attack'] / 10).corr(poke['Defense']), 3))

r(Attack, Defense):     0.439
r(Attack/10, Defense):  0.439


Same answer either way. A rough guide for reading |r|:

| \|r\| | Strength |
|---|---|
| below 0.1 | negligible |
| 0.1 – 0.3 | weak |
| 0.3 – 0.5 | moderate |
| 0.5 – 0.7 | strong |
| above 0.7 | very strong |

The sign gives the direction. Pearson's r measures **straight-line** (linear) relationships only. Hold on to that for section 5.

## 3. A Whole Correlation Matrix
`df.corr()` computes r for **every pair** of columns at once:

In [4]:
corr = poke[stats].corr()
corr.round(2)

,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed
HP,1.00,0.42,0.24,0.36,0.38,0.18
Attack,0.42,1.00,0.44,0.40,0.26,0.38
Defense,0.24,0.44,1.00,0.22,0.51,0.02
Sp. Atk,0.36,0.40,0.22,1.00,0.51,0.47
Sp. Def,0.38,0.26,0.51,0.51,1.00,0.26
Speed,0.18,0.38,0.02,0.47,0.26,1.00


How to read it:
- The diagonal is always 1 (every column correlates perfectly with itself).
- The table is **symmetric**: row `Attack`, column `Defense` equals row `Defense`, column `Attack`.
- So only the cells **above** the diagonal hold new information.

### Finding the Strongest Pairs
Keep the upper triangle, then reshape the table into one row per pair with `.stack()`:

In [5]:
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

pairs = upper.stack().dropna().rename('r').sort_values(key=abs, ascending=False)
pairs.round(3).head(5)

Defense  Sp. Def    0.511
Sp. Atk  Sp. Def    0.506
         Speed      0.473
Attack   Defense    0.439
HP       Attack     0.422
Name: r, dtype: float64

`np.triu(..., k=1)` marks the cells strictly above the diagonal. `.where()` keeps those and turns the rest into `NaN`. `.stack()` turns the table into one row per (row, column) pair, and `.dropna()` throws away the `NaN` pairs. Sorting with `key=abs` ranks by strength, regardless of sign.

In Module 08 this matrix becomes a heatmap, which makes the pattern visible at a glance.

✅ **Your Turn:** Which stat has the **weakest** relationship with `Speed`? (Hint: `corr['Speed']`, then sort.)

In [6]:
speed_relationships = corr['Speed'].drop('Speed').sort_values(key=abs)
display(speed_relationships.rename('pearson_r'))
print(f'Weakest relationship with Speed: {speed_relationships.index[0]} (r={speed_relationships.iloc[0]:.6f}).')
print('Defense has a negligible linear relationship with Speed; near-zero r does not exclude a nonlinear relationship.')


Defense    0.015227
HP         0.175952
Sp. Def    0.259133
Attack     0.381240
Sp. Atk    0.473018
Name: pearson_r, dtype: float64

Weakest relationship with Speed: Defense (r=0.015227).
Defense has a negligible linear relationship with Speed; near-zero r does not exclude a nonlinear relationship.


## 4. Spearman Correlation: When Order Is What Matters
**Spearman's ρ** (rho) replaces each value with its **rank** (1st, 2nd, 3rd, …) and computes Pearson's r on the ranks. That makes it a good choice when:
1. the data are **ordinal** (notebook 01): ranks are all an ordinal column really has
2. the relationship is **curved but always rising (or always falling)**
3. there are **outliers**: a rank can only be "last," no matter how extreme the value

### Curved but always rising

In [7]:
x = pd.Series(range(1, 21))
y = x ** 3

print('Pearson: ', round(x.corr(y), 3))
print('Spearman:', round(x.corr(y, method='spearman'), 3))

Pearson:  0.922


Spearman: 1.0


`y` rises **every single time** `x` rises, so the ranks line up perfectly (Spearman = 1). Pearson is lower because the points don't lie on a straight line.

### Outliers
One extreme point can create (or destroy) a Pearson correlation:

In [8]:
np.random.seed(21)
a = pd.Series(np.random.normal(50, 10, 30))
b = pd.Series(np.random.normal(50, 10, 30))      # unrelated to a

print('No outlier:   Pearson', round(a.corr(b), 3), '| Spearman', round(a.corr(b, method='spearman'), 3))

a_out = pd.concat([a, pd.Series([300])], ignore_index=True)
b_out = pd.concat([b, pd.Series([300])], ignore_index=True)
print('One outlier:  Pearson', round(a_out.corr(b_out), 3), '| Spearman', round(a_out.corr(b_out, method='spearman'), 3))

No outlier:   Pearson 0.002 | Spearman -0.05
One outlier:  Pearson 0.959 | Spearman 0.049


A single point turned two **unrelated** columns into a "very strong" Pearson correlation. Spearman barely noticed. Notebook 03's lesson applies here too: check for outliers before you trust a Pearson r.

### Ordinal data: Titanic class and fare
`Pclass` is ordinal and `Fare` is very skewed, so Spearman is the better fit:

In [9]:
print('Pearson: ', round(titanic['Pclass'].corr(titanic['Fare']), 3))
print('Spearman:', round(titanic['Pclass'].corr(titanic['Fare'], method='spearman'), 3))

Pearson:  -0.549
Spearman: -0.688


Both are negative (a higher class **number**, meaning a cheaper class, goes with a lower fare). Spearman shows the relationship is stronger than Pearson suggests, once the skew in `Fare` stops distorting the calculation.

| Use | When |
|---|---|
| **Pearson** (`method='pearson'`, the default) | both columns are interval/ratio, the pattern is roughly a straight line, and there are no wild outliers |
| **Spearman** (`method='spearman'`) | ordinal data, curved-but-monotonic patterns, or outliers/skew |

## 5. r ≈ 0 Does Not Mean "No Relationship"
Both methods measure relationships that go **one way**. A relationship that rises and then falls can score zero:

In [10]:
x = pd.Series(np.linspace(-10, 10, 41))
y = x ** 2          # y is completely determined by x

print('Pearson: ', round(x.corr(y), 3))
print('Spearman:', round(x.corr(y, method='spearman'), 3))

Pearson:  0.0
Spearman: 0.0


`y` is a perfect function of `x`, and both correlations are 0. Correlation can tell you a relationship **exists**. It can't promise that one **doesn't**.

## 6. Correlation Is Not Causation
Titanic `Fare` and `Survived` are positively correlated:

In [11]:
print('Overall r(Fare, Survived):', round(titanic['Fare'].corr(titanic['Survived']), 3))

Overall r(Fare, Survived): 0.257


Did buying an expensive ticket *cause* survival? Notebook 04 showed that **class** drives both: 1st class paid more **and** survived more. Check the correlation *within* each class:

In [12]:
for cls in [1, 2, 3]:
    in_class = titanic[titanic['Pclass'] == cls]
    r = in_class['Fare'].corr(in_class['Survived'])
    print(f'Class {cls}: r = {r:.3f}  (n = {len(in_class)})')

Class 1: r = 0.191  (n = 216)
Class 2: r = 0.099  (n = 184)
Class 3: r = 0.001  (n = 491)


Within classes, the relationship is weaker. Much of the overall correlation came from the **confounder** (class), not from fare itself. A correlation can come from:
- **A causes B**, or **B causes A** (reverse causation)
- **C causes both** (confounding, like class here)
- **Coincidence**, especially when you test many pairs and report the strongest one

A correlation is a reason to ask **why**. It isn't the answer.

## 7. Same Numbers, Different Data: Anscombe's Quartet
`anscombe.csv` holds four small datasets, `I` through `IV`, each with 11 `(x, y)` pairs. Summarize each one with everything from this module:

In [13]:
anscombe = pd.read_csv('anscombe.csv')

summary = anscombe.groupby('dataset').agg(
    x_mean=('x', 'mean'), x_std=('x', 'std'),
    y_mean=('y', 'mean'), y_std=('y', 'std'),
)
summary['r'] = anscombe.groupby('dataset').apply(lambda d: d['x'].corr(d['y']))
summary.round(2)

,x_mean,x_std,y_mean,y_std,r
dataset,,,,,
I,9.0,3.32,7.5,2.03,0.82
II,9.0,3.32,7.5,2.03,0.82
III,9.0,3.32,7.5,2.03,0.82
IV,9.0,3.32,7.5,2.03,0.82


The four datasets are **statistically identical**: the same means, the same standard deviations, and the same correlation (0.82, "very strong"). By the numbers, they're the same dataset four times.

They aren't. One is a noisy straight line. One is a perfect curve. One is a perfect line with a single outlier. One is a vertical stack of points, with one far-off point creating the entire correlation. You can see some of it by printing the raw values, but that doesn't scale past 11 rows:

In [14]:
anscombe[anscombe['dataset'] == 'IV']

,dataset,x,y
33,IV,8,6.58
34,IV,8,5.76
35,IV,8,7.71
36,IV,8,8.84
37,IV,8,8.47
38,IV,8,7.04
39,IV,8,5.25
40,IV,19,12.50
41,IV,8,5.56
42,IV,8,7.91


> 🎯 **Summary statistics compress data, and compression throws information away.** Notebooks 01–05 gave you the numbers. Module 08 opens by plotting these four datasets, because the only reliable way to see what the numbers hide is to **look**.

---
### 📝 Summary
- Correlation r runs from −1 to 1. The sign is the direction, and |r| is the strength.
- `.corr()` for one pair, `df.corr()` for a matrix. Use the upper triangle to list unique pairs.
- **Spearman** ranks first: use it for ordinal data, curved-but-monotonic patterns, and outliers.
- r ≈ 0 rules out a **straight-line** relationship only.
- Correlation isn't causation. Look for confounders, and compare within groups.
- Identical statistics can hide completely different data (Anscombe). Always plot.

## 🏋️ Practice

### Practice 1 — Total and Its Parts (easy)
`Total` is the sum of the six stats. Compute the correlation of `Total` with each of the six stats, sorted from strongest to weakest. Which stat is most closely tied to `Total`?

In [15]:
assert poke['Total'].eq(poke[stats].sum(axis=1)).all()
total_correlations = poke[stats].corrwith(poke['Total']).sort_values(key=abs, ascending=False)
display(total_correlations.rename('r_with_Total'))
print(f'Most closely tied to Total: {total_correlations.index[0]} (r={total_correlations.iloc[0]:.6f}).')
print('Sp. Atk leads. These are part-whole associations: each stat is included in Total, so correlation is partly built into its definition.')


Sp. Atk    0.747250
Attack     0.736211
Sp. Def    0.717609
HP         0.618748
Defense    0.612787
Speed      0.575943
Name: r_with_Total, dtype: float64

Most closely tied to Total: Sp. Atk (r=0.747250).
Sp. Atk leads. These are part-whole associations: each stat is included in Total, so correlation is partly built into its definition.


### Practice 2 — Pearson vs. Spearman Table (medium)
Build a DataFrame with one row per pair of the six stats and two columns, `pearson` and `spearman` (hint: compute both matrices and stack each upper triangle). Which pair has the biggest difference between the two methods? What might cause that?

In [16]:
pearson_matrix = poke[stats].corr(method='pearson')
spearman_matrix = poke[stats].corr(method='spearman')
upper_mask = np.triu(np.ones(pearson_matrix.shape, dtype=bool), k=1)
method_comparison = pd.concat([
    pearson_matrix.where(upper_mask).stack().dropna().rename('pearson'),
    spearman_matrix.where(upper_mask).stack().dropna().rename('spearman'),
], axis=1)
method_gaps = (method_comparison['pearson'] - method_comparison['spearman']).abs()
method_comparison = method_comparison.loc[method_gaps.sort_values(ascending=False).index]
display(method_comparison.round(6))
largest_gap_pair = method_gaps.idxmax()
print('Largest absolute gap:', largest_gap_pair, round(method_gaps.max(), 6))
display(poke.nlargest(5, 'HP')[['Name', 'HP', 'Defense']])
assert len(method_comparison) == 15  # 6 choose 2; no diagonal or mirrored duplicates.


pearson  spearman
HP      Defense  0.239622  0.433402
        Attack   0.422386  0.565734
        Sp. Def  0.378718  0.493280
        Sp. Atk  0.362380  0.470813
Defense Sp. Atk  0.223549  0.314217
HP      Speed    0.175952  0.265920
Defense Speed    0.015227  0.092818
Attack  Defense  0.438687  0.514995
Defense Sp. Def  0.510747  0.578816
Sp. Atk Sp. Def  0.506121  0.571757
Sp. Def Speed    0.259133  0.320649
Attack  Sp. Def  0.263990  0.320620
        Sp. Atk  0.396362  0.361616
Sp. Atk Speed    0.473018  0.460220
Attack  Speed    0.381240  0.373443

Largest absolute gap: ('HP', 'Defense') 0.193779


,Name,HP,Defense
261,Blissey,255,10
121,Chansey,250,5
217,Wobbuffet,190,58
351,Wailord,170,45
655,Alomomola,165,80


**HP and Defense** have the largest absolute difference: Pearson **0.239622**, Spearman **0.433402**, a gap of **0.193779**. Pearson measures linear association using raw distances, whereas Spearman correlates ranks with average ranks for ties; skew, unusual values, and a curved monotonic pattern can therefore make them differ. In this file, Blissey (HP 255, Defense 10) and Chansey (HP 250, Defense 5) are especially high-HP, low-Defense observations that can pull Pearson downward; the rank measure limits the influence of how extreme their HP values are. This is evidence to inspect the joint distribution, not proof of a particular curve or a reason to delete those Pokemon.


### Practice 3 — Does the Correlation Hold Within Groups? (harder)
Compute r(`Attack`, `Defense`) for all Pokémon, then separately for Legendary and non-Legendary Pokémon, and then for each `Generation`. Put the results in a small table with the group size. Is the relationship consistent across groups?

In [17]:
group_rows = [{'group': 'All Pokemon', 'n': len(poke),
               'pearson_r': poke['Attack'].corr(poke['Defense'])}]
for legendary, group in poke.groupby('Legendary'):
    group_rows.append({'group': f'Legendary={legendary}', 'n': len(group),
                       'pearson_r': group['Attack'].corr(group['Defense'])})
for generation, group in poke.groupby('Generation'):
    group_rows.append({'group': f'Generation {generation}', 'n': len(group),
                       'pearson_r': group['Attack'].corr(group['Defense'])})
attack_defense_groups = pd.DataFrame(group_rows)
display(attack_defense_groups.round(6))
print('The relationship is not consistent across all groups: Legendary Pokemon have a weak negative correlation (-0.211),')
print('while non-Legendary Pokemon have a moderate positive correlation (0.437), close to the overall 0.439.')
print('Every generation has a positive correlation, ranging from 0.357 (Generation 4) to 0.508 (Generation 2).')
print('The Legendary group is smaller (65 versus 735); group mix and sampling variability matter, and correlation is not causal.')


,group,n,pearson_r
0,All Pokemon,800,0.438687
1,Legendary=False,735,0.437192
2,Legendary=True,65,-0.210974
3,Generation 1,166,0.476673
4,Generation 2,106,0.508471
5,Generation 3,160,0.438647
6,Generation 4,121,0.356662
7,Generation 5,165,0.402682
8,Generation 6,82,0.484590


The relationship is not consistent across all groups: Legendary Pokemon have a weak negative correlation (-0.211),
while non-Legendary Pokemon have a moderate positive correlation (0.437), close to the overall 0.439.
Every generation has a positive correlation, ranging from 0.357 (Generation 4) to 0.508 (Generation 2).
The Legendary group is smaller (65 versus 735); group mix and sampling variability matter, and correlation is not causal.


## ✏️ Your Turn
Compute the correlation matrix for `titanic`'s numeric columns (`numeric_only=True`). List the three strongest pairs. For **one** of them, write two sentences: one on what the correlation says, and one on a possible confounder or reason it might not be causal.

In [18]:
titanic_numeric_corr = titanic.corr(numeric_only=True)
titanic_upper = titanic_numeric_corr.where(np.triu(np.ones(titanic_numeric_corr.shape, dtype=bool), k=1))
titanic_pairs = titanic_upper.stack().dropna().rename('pearson_r').sort_values(key=abs, ascending=False)
strongest_three = titanic_pairs.head(3).to_frame()
numeric = titanic.select_dtypes(include='number')
strongest_three['complete_pairs'] = [titanic[[left, right]].dropna().shape[0]
                                    for left, right in strongest_three.index]
display(strongest_three.round(6))
print('All numeric columns are included as requested, including the PassengerId identifier; an identifier correlation has no intrinsic substantive meaning.')
print('Pandas uses complete pairs for each correlation: Age has 177 missing entries, so Pclass-Age uses 714 records.')


,,pearson_r,complete_pairs
Pclass,Fare,-0.549500,891
SibSp,Parch,0.414838,891
Pclass,Age,-0.369226,714


All numeric columns are included as requested, including the PassengerId identifier; an identifier correlation has no intrinsic substantive meaning.
Pandas uses complete pairs for each correlation: Age has 177 missing entries, so Pclass-Age uses 714 records.


The strongest pair, **Pclass and Fare**, has Pearson r = **-0.549500**, so higher class numbers (lower travel classes) tend to have lower recorded fares. This does not establish that changing the class number itself causes a fare change: cabin/ticket arrangements, route, and group-ticket pricing may influence the recorded fare, and class and fare are jointly determined by the travel product purchased.


### Verification of required work

These checks independently parse the supplied local data and recompute Pearson correlations and Spearman correlations using average ranks for ties. The original inputs remain unchanged.


In [19]:
# Independently validate all completed correlation calculations using CSV + statistics.
import csv
import math
import statistics
from itertools import combinations

def read_records(filename):
    with open(filename, newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

poke_records = read_records('pokemon.csv')
titanic_records = read_records('titanic.csv')
anscombe_records = read_records('anscombe.csv')

def average_ranks(values):
    order = sorted(range(len(values)), key=values.__getitem__)
    ranks = [0.0] * len(values)
    start = 0
    while start < len(order):
        end = start + 1
        while end < len(order) and values[order[end]] == values[order[start]]:
            end += 1
        rank = ((start + 1) + end) / 2
        for index in order[start:end]:
            ranks[index] = rank
        start = end
    return ranks

def independent_corr(records, left, right, spearman=False):
    pairs = [(float(row[left]), float(row[right])) for row in records
             if row[left] != '' and row[right] != '']
    first, second = map(list, zip(*pairs))
    if spearman:
        first, second = average_ranks(first), average_ranks(second)
    return statistics.correlation(first, second), len(pairs)

def close(actual, expected):
    assert math.isclose(float(actual), expected, rel_tol=1e-10, abs_tol=1e-12), (actual, expected)

assert len(poke_records) == 800 and len(titanic_records) == 891
assert all(row[column] != '' for row in poke_records for column in stats)
for left, right in combinations(stats, 2):
    pearson_ref, count = independent_corr(poke_records, left, right)
    spearman_ref, _ = independent_corr(poke_records, left, right, spearman=True)
    close(pearson_matrix.loc[left, right], pearson_ref)
    close(spearman_matrix.loc[left, right], spearman_ref)
    assert count == 800
    close(method_comparison.loc[(left, right), 'pearson'], pearson_ref)
    close(method_comparison.loc[(left, right), 'spearman'], spearman_ref)
assert len(pairs) == len(method_comparison) == 15
assert speed_relationships.index[0] == 'Defense' and largest_gap_pair == ('HP', 'Defense')
for column in stats:
    close(total_correlations[column], independent_corr(poke_records, column, 'Total')[0])
for row in attack_defense_groups.itertuples(index=False):
    if row.group == 'All Pokemon':
        records = poke_records
    elif row.group.startswith('Legendary='):
        records = [record for record in poke_records if record['Legendary'].lower() == row.group.split('=')[1].lower()]
    else:
        records = [record for record in poke_records if record['Generation'] == row.group.split()[-1]]
    expected_r, count = independent_corr(records, 'Attack', 'Defense')
    assert row.n == count
    close(row.pearson_r, expected_r)
independent_titanic_pairs = []
for left, right in combinations(titanic_numeric_corr.columns, 2):
    r_ref, count = independent_corr(titanic_records, left, right)
    close(titanic_numeric_corr.loc[left, right], r_ref)
    independent_titanic_pairs.append((left, right, r_ref, count))
independent_titanic_pairs.sort(key=lambda row: abs(row[2]), reverse=True)
assert strongest_three.index.tolist() == [(row[0], row[1]) for row in independent_titanic_pairs[:3]]
for (left, right), row in strongest_three.iterrows():
    expected_r, count = independent_corr(titanic_records, left, right)
    close(row['pearson_r'], expected_r)
    assert row['complete_pairs'] == count
close(titanic['Pclass'].corr(titanic['Fare'], method='spearman'),
      independent_corr(titanic_records, 'Pclass', 'Fare', spearman=True)[0])
close(titanic['Fare'].corr(titanic['Survived']), independent_corr(titanic_records, 'Fare', 'Survived')[0])
for cls in [1, 2, 3]:
    records = [record for record in titanic_records if int(record['Pclass']) == cls]
    group = titanic.loc[titanic['Pclass'] == cls]
    close(group['Fare'].corr(group['Survived']), independent_corr(records, 'Fare', 'Survived')[0])
for name in ['I', 'II', 'III', 'IV']:
    records = [record for record in anscombe_records if record['dataset'] == name]
    close(summary.loc[name, 'r'], independent_corr(records, 'x', 'y')[0])
    assert len(records) == 11
# Verify instructor demonstrations: monotonic curvature, nonmonotonicity, and leverage.
cube_x = list(range(1, 21)); cube_y = [value ** 3 for value in cube_x]
close(pd.Series(cube_x).corr(pd.Series(cube_y)), statistics.correlation(cube_x, cube_y))
close(pd.Series(cube_x).corr(pd.Series(cube_y), method='spearman'), 1.0)
close(x.corr(y), 0.0)
close(x.corr(y, method='spearman'), 0.0)
close(a.corr(b), statistics.correlation(a.tolist(), b.tolist()))
close(a_out.corr(b_out), statistics.correlation(a_out.tolist(), b_out.tolist()))
close(a.corr(b, method='spearman'), statistics.correlation(average_ranks(a.tolist()), average_ranks(b.tolist())))
close(a_out.corr(b_out, method='spearman'), statistics.correlation(average_ranks(a_out.tolist()), average_ranks(b_out.tolist())))
pd.testing.assert_frame_equal(poke, pd.read_csv('pokemon.csv'))
pd.testing.assert_frame_equal(titanic, pd.read_csv('titanic.csv'))
print('PASS: independent Pearson, tied-rank Spearman, 15 unique stat pairs, strongest Titanic pairs, group comparisons, and instructor demonstrations.')
print('Missing numeric values use pairwise deletion; Age pairs have n=714, complete Fare/Pclass pairs n=891.')
print('Anscombe correlations match only approximately; rounded summaries conceal shape and leverage differences.')


PASS: independent Pearson, tied-rank Spearman, 15 unique stat pairs, strongest Titanic pairs, group comparisons, and instructor demonstrations.
Missing numeric values use pairwise deletion; Age pairs have n=714, complete Fare/Pclass pairs n=891.
Anscombe correlations match only approximately; rounded summaries conceal shape and leverage differences.


## 🔥 Challenge (Optional) — Spurious Correlations from Noise
Generate a DataFrame of 20 columns of **pure random noise**, 30 rows each (`np.random.normal(size=(30, 20))`). Compute its correlation matrix and find the strongest pair. How strong is it? What does that say about scanning a big table for the "best" correlation and reporting only that one?

In [20]:
# Optional challenge: scan independent Gaussian noise, with a fixed seed.
# A local generator leaves the required demonstrations' random state untouched.
noise_rng = np.random.default_rng(3603)
noise_columns = [f'noise_{number:02d}' for number in range(1, 21)]
noise_frame = pd.DataFrame(noise_rng.normal(size=(30, 20)), columns=noise_columns)
noise_correlations = noise_frame.corr(method='pearson')
noise_upper = np.triu(np.ones(noise_correlations.shape, dtype=bool), k=1)
noise_pairs = noise_correlations.where(noise_upper).stack().dropna().rename('pearson_r')
noise_pairs = noise_pairs.sort_values(key=abs, ascending=False)
noise_winner = noise_pairs.index[0]
noise_best_r = float(noise_pairs.iloc[0])
print('Independent standard-normal data; seed=3603; shape:', noise_frame.shape)
print('Unique pairs examined:', len(noise_pairs))
print('Strongest pair:', noise_winner)
print(f'Signed Pearson r = {noise_best_r:.6f}; absolute r = {abs(noise_best_r):.6f}')
display(noise_pairs.head(5).to_frame())

# Independently check every pair using Python statistics, including the winner.
assert noise_frame.shape == (30, 20)
assert noise_frame.notna().all().all()
assert len(noise_pairs) == 20 * 19 // 2 == 190
assert all(left != right for left, right in noise_pairs.index)
assert len({frozenset(pair) for pair in noise_pairs.index}) == 190
assert np.allclose(noise_correlations, noise_correlations.T)
assert np.allclose(np.diag(noise_correlations), 1)
noise_references = {}
for left, right in combinations(noise_columns, 2):
    reference = statistics.correlation(noise_frame[left].tolist(), noise_frame[right].tolist())
    noise_references[(left, right)] = reference
    assert math.isclose(noise_pairs.loc[(left, right)], reference, rel_tol=1e-12, abs_tol=1e-12)
assert max(noise_references, key=lambda pair: abs(noise_references[pair])) == noise_winner
assert math.isclose(noise_best_r, noise_references[noise_winner], rel_tol=1e-12, abs_tol=1e-12)
print('PASS: 190 unique pairs, no self/mirrored pairs, and independently verified strongest absolute correlation.')


Independent standard-normal data; seed=3603; shape: (30, 20)
Unique pairs examined: 190
Strongest pair: ('noise_12', 'noise_15')
Signed Pearson r = 0.550690; absolute r = 0.550690


,,pearson_r
noise_12,noise_15,0.550690
noise_09,noise_11,-0.522832
noise_04,noise_20,0.494835
noise_11,noise_19,-0.488654
noise_02,noise_06,0.478942


PASS: 190 unique pairs, no self/mirrored pairs, and independently verified strongest absolute correlation.


**Optional challenge interpretation:** With seed 3603, the strongest of **190** unique pairs is **noise_12 / noise_15**, with Pearson r = **0.550690** (absolute r = **0.550690**) from only **30 observations**. The columns were generated independently, so their population correlations are zero; finite samples still produce nonzero sample correlations, and selecting the largest magnifies the apparent pattern.

Reporting only this winner hides the 189 other comparisons and the search that selected it. The result illustrates selection bias from scanning many pairs, not evidence of a real relationship or cause. Report the full search and sample size, inspect the data, and test a preselected relationship on independent data; if making formal significance claims, account for multiple comparisons. A different seed can produce a different winner and strength; this fixed seed makes the demonstration reproducible.

---
✅ Next up → [Describing Data Lab](06-Describing_Data_Lab.ipynb): put all five notebooks together on a dataset you haven't seen yet.